# Data Loading


This notebook is a placeholder for data loading on the Yelp dataset.

## 1. Standard Imports

In [1]:
print('Kernel Working')

Kernel Working


In [3]:
import pandas as pd
import os

print('Imported')

Imported


## 2. Paths

In [4]:

RAW_DIR = "../data/raw/yelp_json"
INTERIM_DIR = "data/interim"
os.makedirs(INTERIM_DIR, exist_ok=True)

files = {
    "business": "yelp_academic_dataset_business.json",
    "checkin":  "yelp_academic_dataset_checkin.json",
    "review":   "yelp_academic_dataset_review.json",
    "tip":      "yelp_academic_dataset_tip.json",
    "user":     "yelp_academic_dataset_user.json",
}

## 3. File Loading

In [5]:
def load_small_json(filename):
    path = os.path.join(RAW_DIR, filename)
    return pd.read_json(path, lines=True)

df_business = load_small_json(files["business"])
df_checkin  = load_small_json(files["checkin"])
df_tip      = load_small_json(files["tip"])

print(df_business.shape, df_checkin.shape, df_tip.shape)
df_business.head()


(150346, 14) (131930, 2) (908915, 5)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [6]:
df_tip.head()

,user_id,business_id,text,date,compliment_count
0,AGNUgVwnZUey3gcPCJ76iw,3uLgwr0qeCNMjKenHJwPGQ,Avengers time with the ladies.,2012-05-18 02:17:21,0
1,NBN4MgHP9D3cw--SnauTkA,QoezRbYQncpRqyrLH6Iqjg,They have lots of good deserts and tasty cuban...,2013-02-05 18:35:10,0
2,-copOvldyKh1qr-vzkDEvw,MYoRNLb5chwjQe3c_k37Gg,It's open even when you think it isn't,2013-08-18 00:56:08,0
3,FjMQVZjSqY8syIO-53KFKw,hV-bABTK-glh5wj31ps_Jw,Very decent fried chicken,2017-06-27 23:05:38,0
4,ld0AperBXk1h6UbqmM80zw,_uN0OudeJ3Zl_tf6nxg5ww,Appetizers.. platter special for lunch,2012-10-06 19:43:09,0


## 4. Fastparquet Installation and Pandas Version Checking

In [7]:
!pip install fastparquet


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
pip show pandas

Name: pandas
Version: 2.1.4
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
All rights reserved.

Copyright (c) 2011-2023, Open source contributors.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

* Neither the name of the copyright holder nor the names of its
  contributors may be u

In [9]:
def convert_large_json_to_parquet(filename, out_name, chunksize=100_000):
    path = os.path.join(RAW_DIR, filename)
    out_path = os.path.join(INTERIM_DIR, f"{out_name}.parquet")

    reader = pd.read_json(path, lines=True, chunksize=chunksize)
    
    for i, chunk in enumerate(reader):
        if i == 0:
            chunk.to_parquet(out_path, engine="fastparquet")
        else:
            chunk.to_parquet(out_path, engine="fastparquet", append=True)
        print(f"Processed chunk {i+1} ({len(chunk)} rows)")
    
    print(f"Saved: {out_path}")

# Run once — converts large JSON to Parquet for fast future reloads
convert_large_json_to_parquet(files["review"], "review", chunksize=100_000)
convert_large_json_to_parquet(files["user"], "user", chunksize=100_000)

Processed chunk 1 (100000 rows)
Processed chunk 2 (100000 rows)
Processed chunk 3 (100000 rows)
Processed chunk 4 (100000 rows)
Processed chunk 5 (100000 rows)
Processed chunk 6 (100000 rows)
Processed chunk 7 (100000 rows)
Processed chunk 8 (100000 rows)
Processed chunk 9 (100000 rows)
Processed chunk 10 (100000 rows)
Processed chunk 11 (100000 rows)
Processed chunk 12 (100000 rows)
Processed chunk 13 (100000 rows)
Processed chunk 14 (100000 rows)
Processed chunk 15 (100000 rows)
Processed chunk 16 (100000 rows)
Processed chunk 17 (100000 rows)
Processed chunk 18 (100000 rows)
Processed chunk 19 (100000 rows)
Processed chunk 20 (100000 rows)
Processed chunk 21 (100000 rows)
Processed chunk 22 (100000 rows)
Processed chunk 23 (100000 rows)
Processed chunk 24 (100000 rows)
Processed chunk 25 (100000 rows)
Processed chunk 26 (100000 rows)
Processed chunk 27 (100000 rows)
Processed chunk 28 (100000 rows)
Processed chunk 29 (100000 rows)
Processed chunk 30 (100000 rows)
Processed chunk 31 

In [11]:
df_review = pd.read_parquet(os.path.join(INTERIM_DIR, "review.parquet"))
df_user   = pd.read_parquet(os.path.join(INTERIM_DIR, "user.parquet"))

print(df_review.shape, df_user.shape)

(6990280, 9) (1987897, 22)


## 5. Sanity Check Across All Tables

In [12]:
for name, df in [("business", df_business), ("checkin", df_checkin),
                  ("tip", df_tip), ("review", df_review), ("user", df_user)]:
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} cols")

business: 150,346 rows, 14 cols
checkin: 131,930 rows, 2 cols
tip: 908,915 rows, 5 cols
review: 6,990,280 rows, 9 cols
user: 1,987,897 rows, 22 cols
